# 7.5 分类数据

## 7.5.1 背景和目的

In [1]:
import pandas as pd
import numpy as np
values = pd.Series(['apple','orange','apple','apple']*2)
values

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: str

In [2]:
pd.unique(values)

<StringArray>
['apple', 'orange']
Length: 2, dtype: str

In [4]:
values.value_counts()

apple     6
orange    2
Name: count, dtype: int64

In [5]:
values = pd.Series([0,1,0,0]*2)
dim = pd.Series(['apple','orange'])
values

0    0
1    1
2    0
3    0
4    0
5    1
6    0
7    0
dtype: int64

In [6]:
dim

0     apple
1    orange
dtype: str

In [7]:
dim.take(values)

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: str

## 7.5.2 pandas的分类扩展类型

In [10]:
fruits = ['apple','orange','apple','apple']*2
N = len(fruits)
np.random.seed(12345)
df = pd.DataFrame({
    'fruit':fruits,
    'basket_id':np.arange(N),
    'count':np.random.randint(3,15,size=N),
    'weight':np.random.uniform(0,4,size=N)},
    columns=['basket_id','fruit','count','weight']
)
df

,basket_id,fruit,count,weight
0,0,apple,5,3.858058
1,1,orange,8,2.612708
2,2,apple,4,2.995627
3,3,apple,7,2.614279
4,4,apple,12,2.990859
5,5,orange,8,3.845227
6,6,apple,5,0.033553
7,7,apple,4,0.425778


In [11]:
fruit_cat = df['fruit'].astype('category')
fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, str): ['apple', 'orange']

In [13]:
type(fruit_cat.array)

pandas.Categorical

In [14]:
fruit_cat.array.categories

Index(['apple', 'orange'], dtype='str')

In [15]:
fruit_cat.array.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

In [16]:
dict(enumerate(fruit_cat.array.categories))

{0: 'apple', 1: 'orange'}

In [17]:
df['fruit'] = df['fruit'].astype('category')
df['fruit']

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, str): ['apple', 'orange']

In [18]:
# 直接创建
my_categories = pd.Categorical(['foo','bar','baz','foo','bar'])
my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, str): ['bar', 'baz', 'foo']

In [20]:
categories = ['foo','bar','baz']
codes = [0,1,2,0,0,1]
my_cat_2 = pd.Categorical.from_codes(codes,categories)
my_cat_2

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, str): ['foo', 'bar', 'baz']

## 7.5.3 使用Categorical对象进行计算

In [23]:
rng = np.random.default_rng(12345)
draws = rng.standard_normal(1000)
draws[:5].round(3)

array([-1.424,  1.264, -0.871, -0.259, -0.075])

In [24]:
bins = pd.qcut(draws,4)
bins

[(-3.121, -0.675], (0.687, 3.211], (-3.121, -0.675], (-0.675, 0.0134], (-0.675, 0.0134], ..., (0.0134, 0.687], (0.0134, 0.687], (-0.675, 0.0134], (0.0134, 0.687], (-0.675, 0.0134]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.121, -0.675] < (-0.675, 0.0134] < (0.0134, 0.687] < (0.687, 3.211]]

In [26]:
bins = pd.qcut(draws,4,labels=['Q1','Q2','Q3','Q4'])
bins

['Q1', 'Q4', 'Q1', 'Q2', 'Q2', ..., 'Q3', 'Q3', 'Q2', 'Q3', 'Q2']
Length: 1000
Categories (4, str): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [28]:
bins.codes[:10]

array([0, 3, 0, 1, 1, 0, 0, 2, 2, 0], dtype=int8)

In [31]:
bins = pd.Series(bins,name='quartile')
result = pd.Series(draws).groupby(bins).agg(['count','min','max']).reset_index()
print(result)

  quartile  count       min       max
0       Q1    250 -3.119609 -0.678494
1       Q2    250 -0.673305  0.008009
2       Q3    250  0.018753  0.686183
3       Q4    250  0.688282  3.211418


### 使用分类提高性能

In [32]:
N = 10_000_000
labels = pd.Series(['foo','bar','baz','qux']*(N//4))
categories = labels.astype('category')
labels.memory_usage(deep=True)

520000132

In [33]:
categories.memory_usage(deep=True)

10000340

In [34]:
%time _ = labels.astype('category')

CPU times: total: 297 ms
Wall time: 292 ms


In [35]:
# value_counts()内部也使用了groupby机制 所以更快
%timeit labels.value_counts()

269 ms ± 13.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
%timeit categories.value_counts()

39.4 ms ± 579 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## 7.5.4 分类方法

In [37]:
s = pd.Series(['a','b','c','d']*2)
cat_s = s.astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, str): ['a', 'b', 'c', 'd']

In [38]:
cat_s.cat.codes

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

In [39]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='str')

In [40]:
# 假设我们知道这个数据实际分类集合超出了数据中观察到的4个值,则可以使用set_categories方法进行修改
actual_categories = ['a','b','c','d','e']
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, str): ['a', 'b', 'c', 'd', 'e']

In [41]:
cat_s.value_counts()

a    2
b    2
c    2
d    2
Name: count, dtype: int64

In [42]:
cat_s2.value_counts()

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

In [43]:
cat_s3 = cat_s[cat_s.isin(['a','b'])]
cat_s3

0    a
1    b
4    a
5    b
dtype: category
Categories (4, str): ['a', 'b', 'c', 'd']

In [44]:
cat_s3.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (2, str): ['a', 'b']

In [46]:
# add_categories remove_categories
print(pd.get_dummies(cat_s))

,a,b,c,d
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True
4,True,False,False,False
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True


# End